In [1]:
%load_ext dockermagic

# Hive
![Hive](https://hive.apache.org/images/hive_logo_medium.jpg)

- https://hive.apache.org/

## Setup

- version 3.1.3

In [2]:
%%dockerexec hadoop

# Download package
ls /opt/pkgs

apache-hive-3.1.3-bin.tar.gz
apache-hive-4.0.1-bin.tar.gz
hadoop-3.3.6.tar.gz


In [3]:
%%dockerexec hadoop

rm -f /opt/hive

In [4]:
%%dockerexec hadoop

# Download package
mkdir -p /opt/pkgs

cd /opt/pkgs
# wget -q -c https://downloads.apache.org/hive/hive-3.1.3/apache-hive-3.1.3-bin.tar.gz # nao funciona
wget -q -c https://archive.apache.org/dist/hive/hive-3.1.3/apache-hive-3.1.3-bin.tar.gz 

# unpack file and create link
tar -zxf apache-hive-3.1.3-bin.tar.gz -C /opt
ln -s /opt/apache-hive-3.1.3-bin /opt/hive


# update envvars.sh
cat >> /opt/envvars.sh << EOF
# Hive
export HIVE_HOME=/opt/hive
export PATH=\${PATH}:\${HIVE_HOME}/bin

EOF

# Fix slf4j
rm /opt/hive/lib/log4j-slf4j-impl-2.17.1.jar

cat /opt/envvars.sh

export JAVA_HOME=/usr/lib/jvm/java-1.8.0-openjdk-amd64
export PDSH_RCMD_TYPE=ssh

export HADOOP_HOME=/opt/hadoop
export HADOOP_VERSION=3.3.6
export HADOOP_COMMON_HOME=${HADOOP_HOME}
export HADOOP_CONF_DIR=${HADOOP_HOME}/etc/hadoop
export HADOOP_HDFS_HOME=${HADOOP_HOME}
export HADOOP_MAPRED_HOME=${HADOOP_HOME}
export HADOOP_YARN_HOME=${HADOOP_HOME}

export PATH=${PATH}:${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin     

# Hive
export HIVE_HOME=/opt/hive
export PATH=${PATH}:${HIVE_HOME}/bin



## Hadoop configuration (for beeline)

- core-site.xml

```xml
<configuration>
...
<property>
  <name>hadoop.proxyuser.hadoop.groups</name>
  <value>*</value>
</property>
<property>
  <name>hadoop.proxyuser.hadoop.hosts</name>
  <value>*</value>
</property>
</configuration>
```

## Hive Metastore

- using local Derby database

### Create directory in HDFS

In [5]:
%%dockerexec hadoop

source /opt/envvars.sh

hdfs dfs -mkdir -p /user/hive/warehouse
hdfs dfs -chmod g+w /user/hive/warehouse

In [6]:
%%dockerexec hadoop


source /opt/envvars.sh

# hdfs dfs -ls /user/hive

# echo $HIVE_HOME
ls $HIVE_HOME
# mkdir -p /opt/hive/hiveserver2
ls -la /opt/hive


LICENSE
NOTICE
RELEASE_NOTES.txt
bin
binary-package-licenses
conf
examples
hcatalog
jdbc
lib
scripts
lrwxrwxrwx 1 hadoop hadoop 26 May  2 07:35 /opt/hive -> /opt/apache-hive-3.1.3-bin


### Initialize database

In [7]:
%%dockerexec hadoop

source /opt/envvars.sh

mkdir -p $HIVE_HOME/hiveserver2
cd $HIVE_HOME/hiveserver2
$HIVE_HOME/bin/schematool -dbType derby -initSchema 2> /dev/null

Metastore connection URL:	 jdbc:derby:;databaseName=metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
Starting metastore schema initialization to 3.1.0
Initialization script hive-schema-3.1.0.derby.sql
Initialization script completed
schemaTool completed


### Start hiveserver2

In [8]:
%%dockerexec hadoop

source /opt/envvars.sh

cd /opt/hive/hiveserver2
nohup /opt/hive/bin/hive --service hiveserver2 \
--hiveconf hive.security.authorization.createtable.owner.grants=ALL \
--hiveconf hive.root.logger=INFO,console > hiveserver2.out 2>&1 &
echo $! > hiveserver2.pid

## Example

- SF Bay Area Bike Share (https://www.kaggle.com/benhamner/sf-bay-area-bike-share)
- stations.csv and trips.csv

In [9]:
%%dockerexec hadoop

source /opt/envvars.sh

mkdir -p /opt/datasets_hive

In [10]:
%%bash

# copy datasets used by hive examples to hadoop container
docker cp hivedataset.tgz hadoop:/opt/datasets_hive

In [ ]:
%%dockerexec hadoop

source /opt/envvars.sh

cd /opt/datasets_hive
tar -zxf hivedataset.tgz
rm hivedataset.tgz
ls

hdfs dfs -mkdir -p bikeshare/stations
hdfs dfs -put stations.csv bikeshare/stations
hdfs dfs -mkdir -p bikeshare/trips
hdfs dfs -put trips.csv bikeshare/trips

stations.csv
trips.csv


In [13]:
%%bash

# copy inserts gerados
docker cp script_insert_stations.sql hadoop:/opt/script_insert_stations.sql
docker cp sql_insert_trips_100k.sql hadoop:/opt/script_insert_trips.sql

## Using beeline

In [14]:
%%dockerwrite hadoop /opt/script.sql

-- configure jobs executor
SET hive.execution.engine=mr;
SET mapreduce.framework.name=yarn;

-- create bikeshare database
CREATE DATABASE bikeshare;
SHOW DATABASES;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [15]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql
# beeline -n hadoop -u jdbc:hive2://  -f /opt/script.sql








+----------------+
| database_name  |
+----------------+
| bikeshare      |
| default        |
+----------------+


In [18]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

-- create stations table
drop table if exists stations2;
CREATE EXTERNAL TABLE stations2 (
    station_id INT,
    name STRING,
    lat DOUBLE,
    long DOUBLE,
    dockcount INT,
    landmark STRING,
    installation STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 'hdfs:///user/hadoop/bikeshare/stations';

-- create trips table
drop table if exists trips2;
CREATE EXTERNAL TABLE trips2 (
    trip_id INT,
    duration INT,
    start_date STRING,
    start_station STRING,
    start_terminal INT,
    end_date STRING,
    end_station STRING,
    end_terminal INT,
    bike_num INT,
    subscription_type STRING,
    zip_code STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 'hdfs:///user/hadoop/bikeshare/trips';



drop table if exists stations;
CREATE TABLE stations (
    station_id INT,
    name STRING,
    lat DOUBLE,
    long DOUBLE,
    dockcount INT,
    landmark STRING,
    installation STRING
)
stored as ORC;


-- create trips table
drop table if exists trips;
CREATE TABLE trips (
    trip_id INT,
    duration INT,
    start_date STRING,
    start_station_name STRING, --
    start_station_id INT,
    end_date STRING,
    end_station_name STRING,
    end_station_id INT,
    bike_id  INT,
    subscription_type STRING,
    zip_code STRING
)
# partitioned by (end_station_id int)
stored as orc
;

-- show tables
SHOW TABLES;

Successfully copied 3.07kB to hadoop:/opt/script.sql


In [19]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql











































































+-----------------+
|    tab_name     |
+-----------------+
| gender_summary  |
| profiles        |
| school_summary  |
| stations        |
| stations2       |
| status_updates  |
| trips           |
| trips2          |
+-----------------+


In [26]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

LOAD DATA INPATH
    '/user/hadoop/bikeshare/trips/trips.csv'
INTO TABLE stations;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [27]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql






Error: Error while compiling statement: FAILED: SemanticException Unable to load data to destination table. Error: The file that you are trying to load does not match the file format of the destination table. (state=42000,code=40000)


In [23]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_stations.sql
beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_trips.sql

25/05/02 01:36:23 [main]: WARN jdbc.HiveConnection: Failed to connect to localhost:10000
Could not open connection to the HS2 server. Please check the server URI and if the URI is correct, then ask the administrator to check the server status.
Error: Could not open client transport with JDBC Uri: jdbc:hive2://localhost:10000: java.net.ConnectException: Connection refused (Connection refused) (state=08S01,code=0)
25/05/02 01:36:24 [main]: WARN jdbc.HiveConnection: Failed to connect to localhost:10000
Could not open connection to the HS2 server. Please check the server URI and if the URI is correct, then ask the administrator to check the server status.
Error: Could not open client transport with JDBC Uri: jdbc:hive2://localhost:10000: java.net.ConnectException: Connection refused (Connection refused) (state=08S01,code=0)


In [107]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

DESCRIBE FORMATTED stations;
DESCRIBE FORMATTED stations2;
DESCRIBE FORMATTED trips;
DESCRIBE FORMATTED trips2;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [108]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql




+-------------------------------+----------------------------------------------------+-----------------------+
|           col_name            |                     data_type                      |        comment        |
+-------------------------------+----------------------------------------------------+-----------------------+
| # col_name                    | data_type                                          | comment               |
| station_id                    | int                                                |                       |
| name                          | string                                             |                       |
| lat                           | double                                             |                       |
| long                          | double                                             |                       |
| dockcount                     | int                                                |                       

In [155]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

-- query - number of trips per terminal
SELECT start_terminal, start_station, COUNT(1) AS count
FROM trips
GROUP BY start_terminal, start_station
ORDER BY count
DESC LIMIT 10;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [156]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql

25/05/02 00:24:28 [main]: WARN jdbc.HiveConnection: Failed to connect to localhost:10000
Could not open connection to the HS2 server. Please check the server URI and if the URI is correct, then ask the administrator to check the server status.
Error: Could not open client transport with JDBC Uri: jdbc:hive2://localhost:10000: java.net.ConnectException: Connection refused (Connection refused) (state=08S01,code=0)


In [114]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

-- query - number of trips per terminal
SELECT *
FROM trips2
limit 10
;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [115]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql








Error: java.io.IOException: java.lang.RuntimeException: ORC split generation failed with exception: org.apache.orc.FileFormatException: Malformed ORC file hdfs://hadoop:9000/user/hadoop/bikeshare/trips/trips.csv. Invalid postscript. (state=,code=0)


In [ ]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

-- query - join between stations and trips
SELECT t.trip_id, t.duration, t.start_date, s.name, s.lat, s.long, s.landmark
FROM stations s
JOIN trips t ON s.station_id = t.start_terminal
LIMIT 10;

In [ ]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql

In [ ]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

-- create stations table
drop table if exists vendas;
CREATE TABLE vendas (
    --ano INT,
    mes INT,
    fruta STRING
)
PARTITIONED BY(ano int)
STORED AS PARQUET
;

drop table if exists vendas2;
CREATE TABLE vendas2 (
    ano INT,
    mes INT,
    fruta STRING
)
;

-- show tables
SHOW TABLES;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [90]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql
beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas.sql
beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas2.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas2.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas2.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas2.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas.sql
# beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script_insert_vendas2.sql


























+-----------+
| tab_name  |
+-----------+
| stations  |
| trips     |
| vendas    |
| vendas2   |
+-----------+


 


+--------------------------+------------+----------+
|         col_name         | data_type  | comment  |
+--------------------------+------------+----------+
| mes                      | int        |          |
| fruta                    | string     |          |
| ano                      | int        |          |
|                          | NULL       | NULL     |
| # Partition Information  | NULL       | NULL     |
| # col_name               | data_type  | comment  |
| ano                      | int        |          |
+--------------------------+------------+----------+


 


+-----------+------------+----------+
| col_name  | data_type  | comment  |
+-----------+------------+----------+
| ano       | int        |          |
| mes       | int        |          |
| fruta     | string     |          |
+-----------+------------+----------+


In [91]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;


SELECT *
FROM vendas
where fruta='abacaxi'
;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [92]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql








+-------------+---------------+-------------+
| vendas.mes  | vendas.fruta  | vendas.ano  |
+-------------+---------------+-------------+
| 6           | abacaxi       | 2010        |
| 7           | abacaxi       | 2010        |
| 8           | abacaxi       | 2010        |
| 9           | abacaxi       | 2010        |
| 10          | abacaxi       | 2010        |
| 11          | abacaxi       | 2010        |
| 12          | abacaxi       | 2010        |
| 4           | abacaxi       | 2010        |
| 5           | abacaxi       | 2010        |
| 7           | abacaxi       | 2010        |
| 8           | abacaxi       | 2010        |
| 9           | abacaxi       | 2010        |
| 10          | abacaxi       | 2010        |
| 11          | abacaxi       | 2010        |
| 12          | abacaxi       | 2010        |
| 2           | abacaxi       | 2011        |
| 5           | abacaxi       | 2011        |
| 6           | abacaxi       | 2011        |
| 7           | abacaxi    

In [83]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;


SELECT *
FROM vendas2
where fruta='abacaxi'
;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [84]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql








+--------------+--------------+----------------+
| vendas2.ano  | vendas2.mes  | vendas2.fruta  |
+--------------+--------------+----------------+
| 2010         | 6            | abacaxi        |
| 2010         | 7            | abacaxi        |
| 2010         | 8            | abacaxi        |
| 2010         | 9            | abacaxi        |
| 2010         | 10           | abacaxi        |
| 2010         | 11           | abacaxi        |
| 2010         | 12           | abacaxi        |
| 2011         | 2            | abacaxi        |
| 2011         | 5            | abacaxi        |
| 2011         | 6            | abacaxi        |
| 2011         | 7            | abacaxi        |
| 2011         | 8            | abacaxi        |
| 2011         | 9            | abacaxi        |
| 2011         | 10           | abacaxi        |
| 2011         | 11           | abacaxi        |
| 2011         | 12           | abacaxi        |
| 2012         | 2            | abacaxi        |
| 2012       

In [44]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;


SELECT *
FROM vendas
where ano=2025 and mes=1
;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [45]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql








+-------------+---------------+-------------+
| vendas.mes  | vendas.fruta  | vendas.ano  |
+-------------+---------------+-------------+
| 1           | maca          | 2025        |
| 1           | abacaxi       | 2025        |
+-------------+---------------+-------------+


## WordCount using Hive

In [ ]:
%%dockerexec hadoop

source /opt/envvars.sh

mkdir -p /opt/datasets_hive
cd /opt/datasets_hive

wget -q -c https://tinyurl.com/y68jxy7f -O stop-word-list.csv
hdfs dfs -mkdir -p stopwords
hdfs dfs -put stop-word-list.csv stopwords
hdfs dfs -cat stopwords/stop-word-list.csv

# download book "The Complete Works of William Shakespeare, by William Shakespeare" from Gutenberg Project
wget -q -c http://www.gutenberg.org/files/100/100-0.txt -O shakespeare.txt

# create directory in HDFS and put file
hdfs dfs -mkdir -p shakespeare
hdfs dfs -put shakespeare.txt shakespeare
hdfs dfs -ls -h shakespeare

In [ ]:
%%dockerwrite hadoop /opt/wordcount.sql

CREATE TABLE shakespeare_text (line STRING);
LOAD DATA INPATH '/user/hadoop/shakespeare/shakespeare.txt' INTO TABLE shakespeare_text;

CREATE TABLE stopwords (word STRING);
CREATE TABLE tempwords (line STRING);
LOAD DATA INPATH '/user/hadoop/stopwords/stop-word-list.csv' INTO TABLE tempwords;

-- split comma-separated stopwords to rows
INSERT INTO stopwords
SELECT word
FROM tempwords
LATERAL VIEW explode(split(line, ',')) t AS word;
DROP TABLE tempwords;

In [ ]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/wordcount.sql

In [ ]:
%%dockerwrite hadoop /opt/wordcount.sql

SELECT w.word, count(1) AS count
FROM (
    SELECT explode(split(regexp_replace(lower(line), '[^a-z\\s]', ''), '\\s+')) AS word
    FROM shakespeare_text
) w
LEFT OUTER JOIN (
    SELECT lower(trim(word)) AS word
    FROM stopwords
) s ON w.word = s.word
WHERE s.word IS NULL AND w.word != ''
GROUP BY w.word
ORDER BY count DESC
LIMIT 30;

In [ ]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/wordcount.sql

In [157]:
%%dockerexec hadoop

cd /opt/hive/hiveserver2

# kill hiveserver2
kill $(cat hiveserver2.pid)
rm hiveserver2.pid